<img align="left" src="https://docs.linea.org.br/images/logo-linea.fw_2.png" width=100 style="padding: 40px">  
<img align="left" src="https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width=180>

# Photo-z final-product QA
<font size=4>Configuration-driven QA of HATS point estimates and parquet PDFs</font>

This notebook performs a lightweight final-product QA with one distributed pass over the projected HATS columns and one over the projected PDF parquet columns. It checks product integrity, builds global N(z), compares point estimates with stacked PDFs, and reconstructs HATS summaries from a deterministic PDF sample.

Throughout this report, **HATS** means the final `pz_summary` catalog of scalar point estimates, while **PDF parquet** means the separate final product containing one redshift-probability vector per object. The PDF redshift grid comes from that product's `xvals.parquet`.

Paths, cuts, thresholds, and cluster resources come from YAML. This remains a compute and scientific-smoke QA: bias, NMAD, outlier rate, PIT, and coverage require truth redshifts and are intentionally outside its scope.

## 1. Select and inspect the configuration

Set `PZ_QA_CONFIG` before starting Jupyter or change the fallback below. Relative paths in YAML are resolved from the configuration file.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import Markdown, display
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "pz_qa.py").exists():
    NOTEBOOK_DIR = Path("doc/notebooks").resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from pz_qa import load_config, plot_nz, run_qa, start_cluster, write_report

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

CONFIG_PATH = Path(os.environ.get(
    "PZ_QA_CONFIG",
    NOTEBOOK_DIR / "config/dp1-v2-fzboost.yaml",
)).expanduser().resolve()
config = load_config(CONFIG_PATH)
display(Markdown("### Run metadata"))
display(pd.Series(config["run"], name="value").rename_axis("property").to_frame())
display(Markdown("### Science samples included in every result"))
display(pd.DataFrame(config["selections"]).set_index("name"))

## 2. Start Dask

The YAML supports `local`, `slurm`, and `existing`. Adaptive SLURM settings are kept with each run configuration. Cluster information shown below describes computational execution only; it is not a scientific result.

In [ ]:
client, cluster = start_cluster(config)
display(Markdown("**Dask cluster started successfully.**"))
display(pd.Series(config["cluster"], name="value").rename_axis("setting").to_frame())

## 3. Run the distributed QA

Each worker returns only fixed-size histograms, counters, moments, identifier digests, and a small deterministic cross-check sample. Enabled quality selections are computed together, avoiding a new catalog scan per cut.

In [ ]:
result = run_qa(config)
display(Markdown("**Distributed QA completed.** The following sections contain only reduced metrics, checks, and plots; full catalogs and PDF matrices were not collected into the notebook process."))

## 4. Input products and integrity checks

The QA reads two distinct final products:

- **HATS `pz_summary`:** `objectId`, sky coordinates, `zmode`, `zmean`, `zmedian`, `z_p16`, `z_p84`, and columns used by quality selections. HATS supplies the point-estimate statistics, interval diagnostics, coordinates, and point-estimate N(z).
- **PDF parquet product:** `objectId`, the fixed-size `pdf` vectors, and columns used by quality selections. `xvals.parquet` in this directory supplies the redshift grid. The parquet product supplies PDF validity diagnostics, raw PDF sums, stacked-PDF N(z), and reconstructed summaries.

Checks mentioning both products are cross-product checks. The `objectId` comparison uses count plus three commutative 64-bit digests, avoiding a catalog-wide shuffle unless exact uniqueness is enabled in YAML.

In [ ]:
display(Markdown("### Configured inputs"))
paths_table = result.paths.rename_axis("item").reset_index(name="value")
display(paths_table.style.set_properties(
    subset=["value"],
    **{"white-space": "normal", "overflow-wrap": "anywhere", "text-align": "left"},
))

display(Markdown("### Redshift grid — from PDF `xvals.parquet`"))
display(result.grid.rename_axis("property").to_frame())

display(Markdown("### Integrity checks — HATS, PDF parquet, and cross-product"))
display(result.checks.style.map(
    lambda value: "background-color: #d8f3dc" if value else "background-color: #ffccd5",
    subset=["passed"],
))

## 5. Global photo-z metrics by source

The tables below are split by source. HATS point-estimate percentiles are read from global histograms. PDF metrics come from the normalized stack, while raw PDF-sum diagnostics are measured before normalization. Interval-width and PDF-sum quantiles use deterministic bounded samples.

For the global distributions, `count` is the number of finite values; `mean`, `std`, `p16`, `median`, and `p84` summarize redshift; `outside_grid_fraction` measures point estimates outside the configured redshift support; and `edge_fraction` measures occupancy of the first and last grid bins. PDF validity and raw-sum columns diagnose the stored vectors before they are normalized for stacking.

The cross-product shape table compares each HATS point-estimate N(z) with the stacked parquet-PDF N(z). Total variation is dimensionless and ranges from 0 for identical normalized histograms to 1 for disjoint ones; `wasserstein_z` is a redshift-scale displacement. These are consistency diagnostics, not accuracy metrics against truth.

In [ ]:
estimators = result.global_metrics.index.get_level_values("estimator")
hats_metrics = result.global_metrics[estimators != "stacked_pdf"]
pdf_metrics = result.global_metrics[estimators == "stacked_pdf"]

hats_health_columns = [
    "HATS rows", "interval width median (sample)",
    "invalid coordinates", "invalid intervals",
    "non-finite intervals", "intervals outside grid",
]
pdf_health_columns = [
    "PDF rows", "valid PDF fraction", "wrong length", "non-finite",
    "negative", "non-positive sum", "PDF sum mean", "PDF sum std",
    "PDF sum min", "PDF sum max", "PDF sum p01 (sample)",
    "PDF sum median (sample)", "PDF sum p99 (sample)",
]

display(Markdown("### From the HATS `pz_summary` catalog"))
display(hats_metrics)
display(result.pdf_health[hats_health_columns])

display(Markdown("### From the PDF parquet product"))
display(pdf_metrics)
display(result.pdf_health[pdf_health_columns])

display(Markdown("### Cross-product N(z) shape comparison"))
display(result.distribution_distances)

### Cross-product check: HATS summaries reconstructed from parquet PDFs

For a deterministic sample keyed by `objectId`, this check recomputes `zmode`, `zmean`, `zmedian`, `z_p16`, and `z_p84` from the parquet PDF vectors and compares them with the stored HATS columns. These values should agree up to the configured numerical tolerance because Stage 3 derives the HATS summaries from the same PDFs. This is a product-consistency test, not an accuracy measurement against truth.

In [ ]:
display(Markdown("`compared` is the number of matched sampled objects. Absolute differences are in redshift units; `fraction_over_tolerance` uses the tolerance configured in YAML."))
display(result.crosscheck)

## 6. Global N(z)

The left panel contains HATS point-estimate counts. The right panel compares the normalized HATS `zmode`, `zmean`, and `zmedian` shapes with the normalized stack of parquet PDFs for every enabled selection.

In [ ]:
figure = plot_nz(result)
figure

## 7. Save compact reports and release resources

Set `paths.report_dir` in YAML to write CSV/JSON summaries. Full catalogs and PDF matrices are never written or collected here.

In [ ]:
report_dir = write_report(result)
print(f"Report directory: {report_dir}" if report_dir else "Report writing disabled.")
print(f"Checks passed: {int(result.checks['passed'].sum())}/{len(result.checks)}")

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("Dask resources released.")

### Acknowledgments

*This notebook was supported by resources supplied by the Laboratório Interinstitucional de e-Astronomia (LIneA).*  
*These results were obtained using RAIL, an open-source software package made available by the LSST Dark Energy Science Collaboration (DESC).*